In [1]:
import pandas as pd

path = '../../resources/stack-overflow-developer-survey-2025/survey_results_public.csv'
df = pd.read_csv(path)

/tmp/ipykernel_365749/3346430535.py:4: DtypeWarning: Columns (56,74,92,97,98,105,109,110,132,162,165) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


In [2]:
have_worked_with_ai = df['AIModelsHaveWorkedWith'].value_counts()
print(have_worked_with_ai)

AIModelsHaveWorkedWith
openAI GPT (chatbot models)                                                                                                                                                                                                                                                     2379
Anthropic: Claude Sonnet;openAI GPT (chatbot models)                                                                                                                                                                                                                             934
openAI GPT (chatbot models);openAI Reasoning models                                                                                                                                                                                                                              460
Anthropic: Claude Sonnet                                                                                                                          

In [3]:
want_to_work_with_ai = df['AIModelsWantToWorkWith'].value_counts()
print(want_to_work_with_ai)

AIModelsWantToWorkWith
openAI GPT (chatbot models)                                                                                                                                                    1368
Anthropic: Claude Sonnet                                                                                                                                                        600
Anthropic: Claude Sonnet;openAI GPT (chatbot models)                                                                                                                            530
openAI GPT (chatbot models);openAI Reasoning models                                                                                                                             302
openAI GPT (chatbot models);openAI Image generating models;openAI Reasoning models                                                                                              271
                                                                             

In [4]:
group_map = {
    "openAI GPT (chatbot models)": "OpenAI",
    "openAI Image generating models": "OpenAI",
    "openAI Reasoning models": "OpenAI",

    "Anthropic: Claude Sonnet": "Anthropic",
    "Claude": "Anthropic",

    "Gemini (Flash general purpose models)": "Google",
    "Gemini (Pro Reasoning models)": "Google",

    "Meta Llama (all models)": "Meta",
    "Mistral AI models": "Mistral",
    "DeepSeek (R- Reasoning models)": "DeepSeek",
    "DeepSeek (V- General purpose models)": "DeepSeek",

    "Microsoft Phi-4 models": "Microsoft",
    "Perplexity Sonar models": "Perplexity",
    "Alibaba Cloud Qwen models": "Alibaba",

    "X Grok models": "X",
    "Amazon Titan models": "Amazon",
    "Cohere: Command A": "Cohere",
    "Reka (Flash 3 or other Reka models)": "Reka"
}

In [5]:
# pseudocode for plotting the data
""" from itertools import product
import pandas as pd

rows = []

for _, r in df.iterrows():
    for w1, w2 in product(r["worked_with"], r["want_to_work_with"]):
        rows.append((w1, w2))

edges = pd.DataFrame(rows, columns=["worked", "want"]) """

' from itertools import product\nimport pandas as pd\n\nrows = []\n\nfor _, r in df.iterrows():\n    for w1, w2 in product(r["worked_with"], r["want_to_work_with"]):\n        rows.append((w1, w2))\n\nedges = pd.DataFrame(rows, columns=["worked", "want"]) '

In [6]:
df["AIModelsHaveWorkedWith"] = df["AIModelsHaveWorkedWith"].apply(
    lambda x: x.split(";") if isinstance(x, str) else []
)
df["AIModelsWantToWorkWith"] = df["AIModelsWantToWorkWith"].apply(
    lambda x: x.split(";") if isinstance(x, str) else []
)
df["AIModelsHaveWorkedWith"].head()


0    [openAI GPT (chatbot models), openAI Image gen...
1                        [openAI GPT (chatbot models)]
2    [Gemini (Flash general purpose models), openAI...
3                                                   []
4                        [openAI GPT (chatbot models)]
Name: AIModelsHaveWorkedWith, dtype: object

In [7]:
from itertools import product
import pandas as pd

rows = []

for _, r in df.iterrows():
    for a, b in product(r["AIModelsHaveWorkedWith"], r["AIModelsWantToWorkWith"]):
        rows.append((a.strip(), b.strip()))

edges = pd.DataFrame(rows, columns=["worked", "want"])

edges["worked"] = edges["worked"].map(group_map).fillna(edges["worked"])
edges["want"] = edges["want"].map(group_map).fillna(edges["want"])

In [8]:
matrix = pd.crosstab(edges["worked"], edges["want"])
print(matrix)

want        Alibaba  Amazon  Anthropic  Cohere  DeepSeek  Google  Meta  \
worked                                                                   
Alibaba         465      49        418      49       665     647   257   
Amazon           50     121        131      37       140     190    78   
Anthropic       402     169       4767      94      2265    4154  1012   
Cohere           49      37         82      58       115     141    65   
DeepSeek        732     222       2428     163      5342    4208  1291   
Google          669     284       4259     187      3884   10335  1630   
Meta            317     123       1179      89      1372    1872  1419   
Microsoft       135      53        292      48       391     517   234   
Mistral         214      66        716      54       809    1039   492   
OpenAI         1126     553       8677     324      7832   12831  3334   
Perplexity      148      78        665      63       692     946   282   
Reka             28      30         40

In [9]:
heatmap = (
    edges
    .value_counts(["worked", "want"])
    .reset_index(name="value")
)

heatmap.to_json("heatmap.json", orient="records")

In [10]:
# Normalize row-wise (each row sums to 1.0)
# axis=1 calculates the sum across columns for each row
# axis=0 applies the division across the index (rows)
normalized_matrix = matrix.div(matrix.sum(axis=1), axis=0)

print(normalized_matrix)

want         Alibaba    Amazon  Anthropic    Cohere  DeepSeek    Google  \
worked                                                                    
Alibaba     0.113720  0.011983   0.102225  0.011983  0.162631  0.158229   
Amazon      0.037908  0.091736   0.099318  0.028052  0.106141  0.144049   
Anthropic   0.017734  0.007455   0.210296  0.004147  0.099921  0.183254   
Cohere      0.050620  0.038223   0.084711  0.059917  0.118802  0.145661   
DeepSeek    0.029606  0.008979   0.098200  0.006593  0.216057  0.170192   
Google      0.018337  0.007784   0.116736  0.005126  0.106458  0.283275   
Meta        0.028178  0.010933   0.104800  0.007911  0.121956  0.166400   
Microsoft   0.040971  0.016085   0.088619  0.014568  0.118665  0.156904   
Mistral     0.032122  0.009907   0.107475  0.008106  0.121435  0.155959   
OpenAI      0.015156  0.007444   0.116796  0.004361  0.105422  0.172710   
Perplexity  0.025290  0.013329   0.113636  0.010766  0.118250  0.161654   
Reka        0.049470  0.0

In [12]:
# 1. Get the raw counts
heatmap = (
    edges
    .value_counts(["worked", "want"])
    .reset_index(name="value")
)

# 2. Normalize "value" per "worked" model
# We divide each count by the sum of counts for that specific 'worked' model
heatmap["value"] = (
    heatmap["value"] / 
    heatmap.groupby("worked")["value"].transform("sum")
)

# 3. Export to JSON
heatmap.to_json("heatmap_normalised.json", orient="records")